# Week 4b.2: The Customer Support Agent

We'll now build the Husky Tech support agent in LangGraph.

The model in the previous notebook could *ask* for a tool and nothing ran it. 
We'll fix that in two steps: first a node that executes tool calls, then one edge that turns the result into an agent.

Here is where we are headed. Solid arrows are fixed edges; **dashed red arrows are conditional**, decided while the graph runs:

<img src="figures/graph_support_agent.png" width="360" alt="the customer support agent graph">

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The tools and the prompt

Both come from `support_tools.py`: the four tools you wrote in W03b, and the system
prompt that sets the agent's rules. Open that file and read it — the docstrings are
how the model chooses between the tools, and the prompt is where the guardrails live.

In [ ]:
from support_tools import TOOLS, SYSTEM_PROMPT

for t in TOOLS:
    print(f"{t.name:26s} {t.description[:58]}")

print("\nSystem prompt:")
print(SYSTEM_PROMPT)

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

sys_msg = SystemMessage(content=SYSTEM_PROMPT)
model_with_tools = model.bind_tools(TOOLS)

## 2. The assistant node

The assistant node will invoke the llm with tools with system prompt and message.

Note that the prompt is **not** stored in the state — it is added at call time, so the conversation cannot edit it away.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def assistant(state: MessagesState) -> dict:
    context = [sys_msg] + state["messages"]
    response = model_with_tools.invoke(context)
    return {"messages": [response]}

## 3. The tool node

In Week 2b we ran tool calls by hand: read `tool_calls` off the last message, find the
matching function, invoke it, wrap the result in a `ToolMessage`. 

`ToolNode` is that same logic, prebuilt: hand it the list of tools and it handles any call to any of them.

- https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.ToolNode

Define a tool node with: ToolNode([list_of_tools])

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(TOOLS)

## 4. Routing to the tools: `tools_condition`

After the assistant speaks there is exactly one question: did it ask for a tool?

- If yes, run the tools.
- If no, it answered in words, and we are done.

`tools_condition` is that question, prebuilt:
- it reads the last message and returns `"tools"` or `END`.

Let's build a router graph with these:

In [ ]:
from langgraph.prebuilt import tools_condition
from IPython.display import Image, display

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)

# TODO: Build the edges so assistant serves as a router between tool_node and END

router = builder.compile()

display(Image(router.get_graph().draw_mermaid_png()))

In [ ]:
result = router.invoke({"messages": [
    HumanMessage(content="Hello")]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = router.invoke({"messages": [
    HumanMessage(content="Can I return HT-1001?")]})

for m in result["messages"]:
    m.pretty_print()

The tool ran and the result is correct. 

But look at what the customer would actually receive: 
two raw tool outputs and no reply addressed to them. 

Nobody put the pieces together, because after the tools ran the graph stopped.

The model never saw the results. It asked a question and never got an answer.

## 5. The Agent Loop

We can turn the graph into a loop by adding an edge to send the tool results **back to the assistant**.

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)

#TODO: build the edges to turn the graph into a loop

support_agent = builder.compile()
display(Image(support_agent.get_graph().draw_mermaid_png()))

In [ ]:
result = support_agent.invoke({"messages": [
    HumanMessage(content="Hello")]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = support_agent.invoke({"messages": [
    HumanMessage(content="Can I return HT-1001?")]})

for m in result["messages"]:
    m.pretty_print()

The model read both tool results and answered the customer.

That edge is the whole difference between a workflow and an agent. 

The graph now has a **cycle**, so the model can go round as many times as the question needs.

E.g., look something up, read it, decide another tool is required, and only then answer. 

Nothing in our code says how many turns that takes; `tools_condition` ends the loop whenever
the model stops asking for tools.

## 6. Test drives

In [ ]:
from langchain.messages import HumanMessage

questions = [HumanMessage(content="Where is my order HT-1002?"), 
             HumanMessage(content="Can I still return the headphones from order HT-1001?"),
             HumanMessage(content="What are your support hours?"),
             HumanMessage(content="This is the third time I am asking about my missing package and nobody helps. I want to talk to a person.")]

In [ ]:
result = support_agent.invoke({"messages": [questions[0]]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = support_agent.invoke({"messages": [questions[1]]})
print(result["messages"][-1].text)

In [ ]:
result = support_agent.invoke({"messages": [questions[2]]})
print(result["messages"][-1].text)

## 7. Memory

In [ ]:
result = support_agent.invoke({"messages": 
                               [HumanMessage(content="Where is my order HT-1002?")]})
print(result["messages"][-1].text)
print("---")
result = support_agent.invoke({"messages": 
                               [HumanMessage(content="And when will it arrive?")]})
print(result["messages"][-1].text)

### Checkpointer in a Graph

Each `invoke` starts from an empty state, so the agent has no idea what "it" refers to. 

LangGraph fixes this with a **checkpointer**: the graph saves its state after every step, keyed by a `thread_id`, and resumes from it on the next call.

- https://docs.langchain.com/oss/python/langgraph/persistence

In a graph, the checkpointers are added to the `compile()` method call.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

#TODO: compile the graph with the memory checkpointer

config = {"configurable": {"thread_id": "customer-1"}}

In [ ]:
result = support_agent.invoke({"messages": 
                               [HumanMessage(content="Where is my order HT-1002?")]}, config)
print(result["messages"][-1].text)
print("---")
result = support_agent.invoke({"messages": 
                               [HumanMessage(content="And when will it arrive?")]}, config)
print(result["messages"][-1].text)

## 8. Chat with your agent

In [ ]:
from agentui import GradioUI

app = GradioUI(support_agent, config, title="Husky Tech Support")
app.launch()

## 9. Recap

This is the entire customer support agent implemented in LangGraph

In [ ]:
from support_tools import TOOLS, SYSTEM_PROMPT
from langchain.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# whichever model you set up in section 0
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

sys_msg = SystemMessage(content=SYSTEM_PROMPT)
model_with_tools = model.bind_tools(TOOLS)


def assistant(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(TOOLS))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

support_agent = builder.compile(checkpointer=MemorySaver())

# a thread_id is one customer's conversation
config = {"configurable": {"thread_id": "customer-1"}}

result = support_agent.invoke(
    {"messages": [HumanMessage(content="Where is my order HT-1002?")]}, config)
print(result["messages"][-1].text)

result = support_agent.invoke(
    {"messages": [HumanMessage(content="And when will it arrive?")]}, config)
print(result["messages"][-1].text)